Transformer Encoder from Scratch in PyTorch
In this project, I have implemented the Encoder part of the Transformer architecture from scratch using PyTorch, without relying on any built-in Transformer layers to ensure full reproducibility and transparency. The implementation includes:

Custom Encoder Layer with:

A fully manual Multi-Head Self-Attention (MHA) module

A Position-wise Feed Forward Neural Network with ReLU activation

Layer Normalization and residual connections to maintain stability and gradient flow

Embedding Layer to represent input tokens and an optional Positional Encoding module to inject sequential information into the model

Output Layer that maps encoder outputs to predicted token IDs

Training setup using CrossEntropyLoss and Stochastic Gradient Descent (SGD) optimizer

The architecture and training loop are fully configurable through an external configuration file that defines hyperparameters such as embedding dimension, vocabulary size, number of heads, and more. This project is designed to provide a clear and reproducible understanding of the Transformer encoder internals and serves as a solid foundation for building custom NLP models.

# Import

In [ ]:
import torch
from torch import Tensor

import torch.nn as nn
from torch.nn import Parameter
import torch.nn.functional as F
from torch.nn.functional import one_hot

import torch.optim as optim

from  pprint import pprint
from yaml import safe_load
import requests
from io import BytesIO

In [ ]:

config_url = "https://raw.githubusercontent.com/Arunprakash-A/LLM-from-scratch-PyTorch/main/config_files/enc_config.yml"
response = requests.get(config_url)
config = response.content.decode("utf-8")
config = safe_load(config)
pprint(config)

{'input': {'batch_size': 10, 'embed_dim': 32, 'seq_len': 8, 'vocab_size': 10},
 'model': {'d_ff': 128,
           'd_model': 32,
           'dk': 4,
           'dq': 4,
           'dv': 4,
           'n_heads': 8,
           'n_layers': 6}}


In [ ]:
print(response)

<Response [200]>


In [ ]:

vocab_size = config['input']['vocab_size']
batch_size = config['input']['batch_size']
seq_len = config['input']['seq_len']
embed_dim = config['input']['embed_dim']

In [ ]:

data_url = 'https://github.com/Arunprakash-A/LLM-from-scratch-PyTorch/raw/main/config_files/w1_input_tokens'
r = requests.get(data_url)
token_ids = torch.load(BytesIO(r.content))
print(token_ids)

tensor([[5, 7, 5, 6, 3, 8, 7, 5],
        [7, 2, 7, 1, 2, 1, 1, 7],
        [1, 0, 0, 3, 6, 3, 0, 8],
        [5, 0, 2, 8, 6, 5, 5, 3],
        [3, 5, 4, 8, 5, 0, 7, 3],
        [8, 6, 7, 4, 4, 4, 0, 1],
        [5, 8, 1, 0, 1, 1, 0, 3],
        [1, 7, 8, 8, 0, 5, 3, 7],
        [7, 7, 1, 4, 5, 6, 7, 0],
        [1, 7, 2, 8, 3, 0, 0, 4]])


<ipython-input-5-0fefa6fcf57f>:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  token_ids = torch.load(BytesIO(r.content))


# Building the sub-layers

In [ ]:

dq = torch.tensor(config['model']['dq'])
dk = torch.tensor(config['model']['dk'])
dv = torch.tensor(config['model']['dv'])
dmodel = embed_dim
heads = torch.tensor(config['model']['n_heads'])
d_ff = config['model']['d_ff']

In [ ]:
class MHA(nn.Module):

   def __init__(self,dmodel,dq,dk,dv,heads):
    super(MHA,self).__init__()
    self.num_heads = heads
    self.d_model = dmodel
    self.dq = dq
    self.dk = dk
    self.dv = dv

    # Linear layers for queries, keys, and values
    # torch.manual_seed(43)
    self.W_q = nn.Linear(dmodel, dq * heads)
    # torch.manual_seed(44)
    self.W_k = nn.Linear(dmodel, dk * heads)
    # torch.manual_seed(45)
    self.W_v = nn.Linear(dmodel, dv * heads)
    # Linear layer for output
    # torch.manual_seed(46)
    self.W_o = nn.Linear(heads * dv, dmodel)
    self.initialize_weights()

    # your method definitions go here (if you want to)

   def initialize_weights(self):
        # Initialize W_Q with seed 43
        torch.manual_seed(43)
        self.W_q.weight=nn.Parameter(torch.randn(dq*heads,dmodel))
        self.W_q.bias = nn.Parameter(torch.randn(dmodel))

        # Initialize W_K with seed 44
        torch.manual_seed(44)
        self.W_k.weight=nn.Parameter(torch.randn(dk*heads,dmodel))
        self.W_k.bias = nn.Parameter(torch.randn(dmodel))

        # Initialize W_V with seed 45
        torch.manual_seed(45)
        self.W_v.weight=nn.Parameter(torch.randn(dv*heads,dmodel))
        self.W_v.bias = nn.Parameter(torch.randn(dmodel))

        # Initialize W_O with seed 46
        torch.manual_seed(46)
        self.W_o.weight=nn.Parameter(torch.randn(dmodel,dv*heads))
        self.W_o.bias = nn.Parameter(torch.randn(dmodel))
   def forward(self,H=None):
    '''
    Input: Size [BSxTxdmodel]
    Output: Size[BSxTxdmodel]
    '''
    batch_size, seq_len, dmodel = H.size()

    # Linear transformations
    Q = self.W_q(H).view(batch_size, seq_len, self.num_heads, self.dq).transpose(1, 2)  # [BS x heads x T x dq]
    K = self.W_k(H).view(batch_size, seq_len, self.num_heads, self.dk).transpose(1, 2)  # [BS x heads x T x dk]
    V = self.W_v(H).view(batch_size, seq_len, self.num_heads, self.dv).transpose(1, 2)  # [BS x heads x T x dv]

    # Scaled dot-product attention
    scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.dk ** 0.5)  # [BS x heads x T x T]
    attn_weights = F.softmax(scores, dim=-1)  # [BS x heads x T x T]
    output = torch.matmul(attn_weights, V)  # [BS x heads x T x dv]

    # Concatenate heads and apply output linear transformation
    output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.num_heads * self.dv)  # [BS x T x (heads * dv)]
    out = self.W_o(output)  # [BS x T x d_model]

    return out

## Pointwise FFN

* Randomly initialize the parameters using normal distribution with the following seed values
  * $W_{1}:$(seed=47)
  * $W_2:$(seed=48)  

In [ ]:
class FFN(nn.Module):
  def __init__(self,dmodel,d_ff,layer=0):
    super(FFN,self).__init__()
    # First linear layer to expand the dimension
    # torch.manual_seed(47)
    self.fc1 = nn.Linear(dmodel, d_ff)
    # Second linear layer to project it back to the original dimension
    # torch.manual_seed(48)
    self.fc2 = nn.Linear(d_ff, dmodel)
    # ReLU activation
    self.relu = nn.ReLU()
    self.initialize_weights()


  def initialize_weights(self):
        # Initialize fc1 with seed 47
        torch.manual_seed(47)
        self.fc1.weight = nn.Parameter(torch.randn(d_ff, dmodel))  # d_ff as output features, d_model as input features
        self.fc1.bias = nn.Parameter(torch.randn(d_ff))

        # Initialize fc2 with seed 48
        torch.manual_seed(48)
        self.fc2.weight = nn.Parameter(torch.randn(dmodel, d_ff))
        self.fc2.bias = nn.Parameter(torch.randn(dmodel))
  def forward(self,x):
    '''
    input: size [BSxTxdmodel]
    output: size [BSxTxdmodel]
    '''
    return self.fc2(self.relu(self.fc1(x)))

    return out

## Output Layer

* Randomly initialize the linear layer
 * $W_L:$(seed=49)


In [ ]:
class OutputLayer(nn.Module):

  def __init__(self,dmodel,vocab_size):
    super(OutputLayer,self).__init__()
    torch.manual_seed(49)
    self.linear = nn.Linear(dmodel, vocab_size)

  def forward(self,representations):
    '''
    input: size [bsxTxdmodel]
    output: size [bsxTxvocab_size]
    Note: Do not apply the softmax. Just return the output of linear transformation
    '''
    out = self.linear(representations)
    return out

## Encoder Layer

In [ ]:
class EncoderLayer(nn.Module):

  def __init__(self,dmodel,dq,dk,dv,d_ff,heads):
    super(EncoderLayer,self).__init__()
    self.mha = MHA(dmodel,dq,dk,dv,heads)
    self.layer_norm_mha = torch.nn.LayerNorm(dmodel)
    self.layer_norm_ffn = torch.nn.LayerNorm(dmodel)
    self.ffn = FFN(dmodel,d_ff)

  def forward(self,x):

    # do a forward pass
    output=self.mha(x)
    output=self.layer_norm_mha(output)
    output=self.ffn(output)
    out=self.layer_norm_ffn(output)

    return out

## Model with one encoder layer

 * The encoders' forward function accepts the token_ids as input
 * Generate the embeddings for the token ids by initializing the emebedding weights from normal distribution by setting the seed value to 50
 * Use `torch.nn.Embed()` to generate required embeddings

In [ ]:
class Encoder1(nn.Module):

  def __init__(self,vocab_size,embed_dim,dq,dk,dv,d_ff,heads,num_layers=1):
    super(Encoder1,self).__init__()
    self.embedding = nn.Embedding(vocab_size, embed_dim)  # Embedding layer
    self.layers = nn.ModuleList([EncoderLayer(embed_dim, dq, dk, dv, d_ff, heads) for _ in range(num_layers)])  # List of Encoder layers
    self.output_layer = OutputLayer(embed_dim, vocab_size)  # Output layer


  def forward(self,x):
    '''
    The input should be tokens ids of size [BS,T]
    '''
    out =  self.embedding(x) # get the embeddings of the tokens
    for layer in self.layers:
            out = layer(out)  # Pass the embeddings through the encoder layers, size: [BS, T, d_model]
    out =  self.output_layer(out)  # Get the logits, size: [BS, T, vocab_size]

    return out

In [ ]:
model = Encoder1(vocab_size,dmodel,dq,dk,dv,d_ff,heads)
optimizer = optim.SGD(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

# Training the model

 * Train the model for 30 epochs and compute the loss

In [ ]:
def train(token_ids,epochs=None):

  for epoch in range(epochs):
    out = model(token_ids)
    loss = criterion(out.view(-1, out.size(-1)), token_ids.view(-1))  # Reshape to [N, C] and [N]
    loss.backward()


    optimizer.step()
    optimizer.zero_grad()
train(token_ids,30)


# Inference

In [ ]:
with torch.inference_mode():
  predictions =  model(token_ids) # predict the labels
  predictions = predictions.argmax(dim=-1)

* See how many labels are correctly predicted

In [ ]:
print(torch.count_nonzero(token_ids==predictions))

tensor(17)


# Encoder with N Layers

  * The intialized parameters in all layers are identical
  * use ModuleList to create **deep-copies** of encoder layer

In [ ]:
import copy

In [ ]:
class Encoder(nn.Module):

  def __init__(self,vocab_size,dmodel,dq,dk,dv,d_ff,heads,num_layers=1):
    super(Encoder,self).__init__()
    self.output_layer = OutputLayer(dmodel, vocab_size)  # Output layer
    self.embed_weights = nn.Embedding(vocab_size, dmodel)  # Embedding layer
    self.enc_layers = nn.ModuleList([self.clone_layer(model) for _ in range(num_layers)])  # List of Encoder layers
    self.output_layer = OutputLayer(embed_dim, vocab_size)  # Output layer
    self.output_layer.linear=copy.deepcopy(model.output_layer.linear)

  def clone_layer(self,layer):
        # Create a new layer and copy parameters from the provided layer
        new_layer = EncoderLayer(dmodel,dq,dk,dv,d_ff,heads)
        # print(new_layer.mha.W_q)
        new_layer.mha.W_q=copy.deepcopy(layer.layers[0].mha.W_q)
        new_layer.mha.W_k=copy.deepcopy(layer.layers[0].mha.W_k)
        new_layer.mha.W_v=copy.deepcopy(layer.layers[0].mha.W_v)
        new_layer.mha.W_o=copy.deepcopy(layer.layers[0].mha.W_o)
        new_layer.ffn.fc1=copy.deepcopy(layer.layers[0].ffn.fc1)
        new_layer.ffn.fc2=copy.deepcopy(layer.layers[0].ffn.fc2)
        new_layer.embedding=copy.deepcopy(layer.embedding)

        return new_layer
  def forward(self,x):
    '''
    1. Get embeddings
    2. Pass it through encoder layer-1 and recursively pass the output to subsequent enc.layers
    3. output the logits
    '''
    out =  self.embed_weights(x) # get the embeddings of the tokens
    for layer in self.enc_layers:
            out = layer(out)  # Pass the embeddings through the encoder layers, size: [BS, T, d_model]
    out =  self.output_layer(out)  # Get the logits, size: [BS, T, vocab_size]
    return out

* Train the stack of encoder layers with `num_layers=2` for the same 30 epochs

In [ ]:
model1 = Encoder(vocab_size,dmodel,dq,dk,dv,d_ff,heads,num_layers=2)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model1.parameters(), lr=0.01)

In [ ]:
def train(token_ids,epochs=30):

  for epoch in range(epochs):
    out = model1(token_ids)
    loss =  criterion(out.view(-1, out.size(-1)), token_ids.view(-1))  # Reshape to [N, C] and [N]
    loss.backward()


    optimizer.step()
    optimizer.zero_grad()


In [ ]:
train(token_ids)

In [ ]:
with torch.inference_mode():
  predictions = predictions =  model1(token_ids) # predict the labels
  predictions = predictions.argmax(dim=-1)

In [ ]:
torch.count_nonzero(predictions==token_ids)

tensor(23)

## Count Number of Parameters

In [ ]:
total_num_parameters = 0
for parameter in model.parameters():
  for parm in parameter:
    total_num_parameters+=parm.numel()


print('total number of parameters in the model \n including the embedding layer is:', total_num_parameters)

total number of parameters in the model 
 including the embedding layer is: 13354
